In [10]:
import fitz
import os
import re
import shutil
import tkinter as tk
import yaml
from datetime import datetime

In [11]:
BASE_DIR      = '/Users/willmino/Library/Claude/Resume_Github_Project'
SRC_PDF       = os.path.join(BASE_DIR, 'Will Mino - Resume.pdf')
YAML_ORIGINAL = os.path.join(BASE_DIR, 'resume_original.yaml')
YAML_WORKING  = os.path.join(BASE_DIR, 'resume.yaml')

In [12]:
CALIBRI_REGULAR = '/Applications/Microsoft Excel.app/Contents/Resources/DFonts/Calibri.ttf'
CALIBRI_ITALIC  = '/Applications/Microsoft Excel.app/Contents/Resources/DFonts/Calibrii.ttf'
CALIBRI_BOLD    = '/Applications/Microsoft Excel.app/Contents/Resources/DFonts/Calibrib.ttf'
ARIAL           = '/System/Library/Fonts/Supplemental/Arial.ttf'

In [13]:
CALIBRI_ASCENDER       = 9.0
CALIBRI_INNER_LINE_HT  = 14.65
CALIBRI_BULLET_ADVANCE = 14.65
ARIAL11_ASCENDER       = 9.958
ARIAL12_ASCENDER       = 10.863

In [14]:
COMPANY_SECTIONS = {
    "truecar": {
        "page": 0, "y_range": (453, 740), "y_start": 465.8,
        "x_text": 54.0, "arial_size": 11, "wrap_width": 522.0, "original_y_end": 740,
    },
    "ekn": {
        "page": 1, "y_range": (86, 300), "y_start": 98.1,
        "x_text": 54.0, "arial_size": 12, "wrap_width": 522.0, "original_y_end": 300,
    },
    "pfizer": {
        "page": 1, "y_range": (345, 545), "y_start": 357.1,
        "x_text": 54.0, "arial_size": 12, "wrap_width": 522.0, "original_y_end": 545,
    },
    "tanabe": {
        "page": 2, "y_range": (86, 200), "y_start": 98.1,
        "x_text": 54.0, "arial_size": 12, "wrap_width": 522.0, "original_y_end": 200,
    },
}

In [15]:
STREAM_PATTERN = re.compile(
    r'\nq\n\.75 0 0 \.75 36 ([\d.]+) cm\n0 0 0 RG 0 0 0 rg\n.*?\nQ',
    re.DOTALL,
)

In [16]:
# ── Parsing ───────────────────────────────────────────────────────────────────

# Each entry: (normalized label, internal key, section type)
# Ordered longest-first so "ekn engineering" matches before "ekn",
# and "core competencies" matches before a hypothetical shorter overlap.
LABEL_MAP = [
    ('technical proficiencies', 'technical_proficiencies', 'rows'),
    ('technical_proficiencies', 'technical_proficiencies', 'rows'),
    ('tech proficiencies',      'technical_proficiencies', 'rows'),
    ('core competencies',       'core_competencies',       'rows'),
    ('core_competencies',       'core_competencies',       'rows'),
    ('ekn engineering',         'ekn',                     'bullets'),
    ('subheader',               'subheader',               'string'),
    ('summary',                 'summary',                 'string'),
    ('truecar',                 'truecar',                 'bullets'),
    ('true car',                'truecar',                 'bullets'),
    ('ekn',                     'ekn',                     'bullets'),
    ('pfizer',                  'pfizer',                  'bullets'),
    ('tanabe',                  'tanabe',                  'bullets'),
]

BULLET_COMPANIES = {'truecar', 'ekn', 'pfizer', 'tanabe'}


def match_label(line):
    """Return (internal_key, section_type, inline_text) if line starts with a known label, else None."""
    normalized = line.lower().strip()
    for label, key, stype in LABEL_MAP:
        prefix = label + ':'
        if normalized.startswith(prefix):
            inline = line[line.lower().find(prefix) + len(prefix):].strip()
            return key, stype, inline
    return None


def parse_updates(text):
    """
    Parse labeled input into a dict of updates.
    Only sections present in the input are included.
    """
    current_key  = None
    current_type = None
    inline       = ''
    body_lines   = []
    updates      = {}

    def flush():
        if current_key is None:
            return
        all_lines = []
        if inline:
            all_lines.append(inline)
        all_lines.extend(ln for ln in body_lines if ln.strip())

        if current_type == 'string':
            updates[current_key] = ' '.join(all_lines)

        elif current_type == 'rows':
            rows = []
            for ln in all_lines:
                items = [x.strip() for x in ln.split(',') if x.strip()]
                if items:
                    rows.append(items)
            updates[current_key] = rows

        elif current_type == 'bullets':
            bullets = []
            for ln in all_lines:
                b = ln.lstrip('-•').strip()
                if b:
                    bullets.append(b)
            updates[current_key] = bullets

    for line in text.split('\n'):
        match = match_label(line)
        if match:
            flush()
            current_key, current_type, inline = match
            body_lines = []
        elif current_key is not None:
            body_lines.append(line)

    flush()
    return updates


def split_even_rows(items, n_rows):
    """Split a flat list of items into n_rows as evenly as possible."""
    total = len(items)
    base, extra = divmod(total, n_rows)
    rows, i = [], 0
    for r in range(n_rows):
        size = base + (1 if r < extra else 0)
        rows.append(items[i:i + size])
        i += size
    return [r for r in rows if r]


def apply_updates(data, updates):
    """Merge parsed updates into the loaded YAML data dict."""
    for key in ('subheader', 'summary', 'core_competencies', 'technical_proficiencies'):
        if key in updates:
            if key == 'technical_proficiencies':
                # Flatten all items then split evenly across 2 rows
                all_items = [item for row in updates[key] for item in row]
                updates[key] = split_even_rows(all_items, 2)
            elif key == 'core_competencies':
                # Flatten all items then enforce exactly 3 per row
                all_items = [item for row in updates[key] for item in row]
                updates[key] = [all_items[i:i+3] for i in range(0, len(all_items), 3)]
            data[key] = updates[key]
            print(f"  Updated: {key}")

    for company in BULLET_COMPANIES:
        if company in updates:
            data['bullets'][company] = updates[company]
            print(f"  Updated: bullets/{company}")

In [ ]:
# ── Main ──────────────────────────────────────────────────────────────────────

INSTRUCTIONS = """
Paste your resume updates below using section labels.
Only the sections you include will be updated — everything else stays the same.
When finished, type END on its own line and press Enter.

  Subheader:               text on the same line
  Summary:                 text on the same line (or next line)
  Core Competencies:       one row per line, items comma-separated (max 4 rows)
  Technical Proficiencies: one row per line, items comma-separated (max 2 rows)
  TrueCar:                 one bullet per line, leading dash optional
  EKN Engineering:         one bullet per line
  Pfizer:                  one bullet per line
  Tanabe:                  one bullet per line

Example:
  Subheader: Senior Data Analyst | 9 Years Experience
  TrueCar:
  - Led pricing analytics for OEM incentive programs.
  - Built dashboards tracking $7M in annual revenue.
──────────────────────────────────────────────────────
"""


def read_input():
    result = []

    root = tk.Tk()
    root.title("Resume Updater")
    root.geometry("700x600")
    root.resizable(True, True)

    # Instructions label at top
    tk.Label(root, text=INSTRUCTIONS, justify="left", font=("Courier", 10),
             anchor="w").pack(fill="x", padx=10, pady=(10, 0))

    # Text box in the middle
    text_box = tk.Text(root, font=("Courier", 12), wrap="word", height=15)
    text_box.pack(fill="both", expand=True, padx=10, pady=10)
    text_box.focus_set()

    # Submit button pinned at the bottom
    def submit():
        result.append(text_box.get("1.0", "end-1c"))
        root.after(1, root.destroy)

    btn = tk.Button(root, text="Submit", font=("Arial", 14, "bold"),
                    bg="#4CAF50", fg="white", height=2, command=submit)
    btn.pack(fill="x", padx=10, pady=(0, 10))

    root.mainloop()
    return result[0] if result else ""


def main():
    data = load_yaml(YAML_WORKING)
    print("Loaded resume.yaml")

    raw = read_input()
    updates = parse_updates(raw)

    if not updates:
        print("\nNo recognized sections found — nothing to update.")
        return

    print("\nApplying updates...")
    apply_updates(data, updates)

    print("\nSaving resume.yaml...")
    save_yaml(YAML_WORKING, data)

    generate_pdf(data)
    print("\nDone.")


if __name__ == "__main__":
    main()

Loaded resume.yaml

Applying updates...
  Updated: subheader
  Updated: summary
  Updated: core_competencies
  Updated: technical_proficiencies
  Updated: bullets/ekn
  Updated: bullets/truecar
  Updated: bullets/pfizer
  Updated: bullets/tanabe

Saving resume.yaml...

Created: Will Mino - Resume_copy_20260327_083943.pdf
Created: resume_copy_20260327_083943.yaml

Removing header section cm blocks (page 0)...
  Subheader cm blocks removed: [99.92]
  Summary cm blocks removed:   [119.45]
  Core Comp cm blocks removed: [255.18, 272.02, 288.87, 305.72]
  Tech Prof cm blocks removed: [356.84, 373.69]

Rendering company bullet sections...
  truecar: removed cm blocks at y=[]
  ekn: removed cm blocks at y=[87.27, 126.57, 155.86, 185.16, 214.46, 243.75, 273.05, 287.7]
  EKN delta=-11.45 — reflowing Pfizer header...
  page 1: shifted y=300–345 by delta=-11.45
  pfizer: removed cm blocks at y=[346.29, 385.59, 414.89, 444.18, 473.48, 502.78, 532.07]
  tanabe: removed cm blocks at y=[87.27, 126.57